# 06 Explainability: SHAP and LIME

Goal: generate global SHAP plots, local explanations, LIME examples, and faithfulness checks.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_preprocessing import TARGET_COLUMN
from explainability import (
    build_lime_explainer,
    compute_shap_values,
    faithfulness_permutation_test,
    save_shap_summary_plots,
    top_shap_features,
)
from model_training import split_features_target

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready_data.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "models" / "final_model.joblib"
SHAP_DIR = PROJECT_ROOT / "outputs" / "shap_results"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
SHAP_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
split = split_features_target(df, target_column=TARGET_COLUMN, test_size=0.2, random_state=42)
final_model = joblib.load(MODEL_PATH)

X_shap, shap_values = compute_shap_values(final_model, split.X_test, max_rows=1000)
save_shap_summary_plots(X_shap, shap_values, SHAP_DIR)
shap_top = top_shap_features(shap_values, top_n=20)
shap_top.to_csv(TABLES_DIR / "shap_top_features.csv", index=False)
shap_top.head(10)

In [ ]:
faithfulness_features = [
    "PAY_0",
    "PAY_2",
    "delay_count",
    "severe_delay_count",
    "LIMIT_BAL",
    "payment_to_bill_ratio",
    "utilization_proxy",
]
faithfulness = faithfulness_permutation_test(final_model, split.X_test, split.y_test, faithfulness_features)
faithfulness.to_csv(TABLES_DIR / "faithfulness_test_results.csv", index=False)
faithfulness

In [ ]:
lime_explainer = build_lime_explainer(split.X_train, categorical_columns=["SEX", "EDUCATION", "MARRIAGE"])
borderline_idx = pd.Series(final_model.predict_proba(split.X_test)[:, 1], index=split.X_test.index).sub(0.5).abs().idxmin()
lime_exp = lime_explainer.explain_instance(
    split.X_test.loc[borderline_idx].to_numpy(),
    final_model.predict_proba,
    num_features=10,
)
lime_exp.save_to_file(str(SHAP_DIR / "lime_borderline_customer.html"))
lime_exp.as_list()